# Phase 1 - Firm Short Reference Workflow

End-to-end walkthrough of the Phase 1 reference implementation for the use case **firm is short on free inventory for delivery**.

Sections:
1. Load CSV data
2. Load YAML configuration
3. Call each CSV-backed tool independently
4. Run the full ADK / local workflow
5. Inspect evidence bundle and diagnosis
6. Read draft commentary
7. Capture human approval
8. Run the eval suite


In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

## 1. Load CSV data

In [ ]:
from settlement_agent.infrastructure import csv_loader
import pandas as pd

positions = pd.DataFrame(csv_loader.load_positions())
settlements = pd.DataFrame(csv_loader.load_settlements())
reference = pd.DataFrame(csv_loader.load_reference())
trade_netting = pd.DataFrame(csv_loader.load_trade_netting())
scenarios = pd.DataFrame(csv_loader.load_scenario_manifest())

print('positions:', positions.shape)
print('settlements:', settlements.shape)
print('reference:', reference.shape)
print('trade_netting:', trade_netting.shape)
scenarios.head()

## 2. Load YAML configuration

In [ ]:
from settlement_agent.utils import yaml_loader

use_case = yaml_loader.load_use_case()
agents = yaml_loader.load_agents()
workflow = yaml_loader.load_workflow()
prompts = yaml_loader.load_prompts()
tools_cfg = yaml_loader.load_tools_config()
policy = yaml_loader.load_policy()
evals = yaml_loader.load_eval_suite()

print('use case:', use_case['use_case_id'], '-', use_case['name'])
print('workflow:', workflow['workflow_id'], 'v' + workflow['workflow_version'])
print('agents:', [a['id'] for a in agents['agents']])
print('tools:', [t['name'] for t in tools_cfg['tools']])
print('policy:', policy['policy_id'])
print('evals:', evals['eval_suite_id'], '(', len(evals['cases']), 'cases )')

validation = yaml_loader.validate_all_configs()
assert all(validation.values()), validation
validation

## 3. Call each CSV-backed tool independently

In [ ]:
from settlement_agent.tools.position_tool import call_position_tool
from settlement_agent.tools.settlement_tool import call_settlement_tool
from settlement_agent.tools.reference_data_tool import call_reference_data_tool
from settlement_agent.tools.trade_netting_tool import call_trade_netting_tool

pos = call_position_tool({'account_id': 'ACC-DLV-001', 'security_id': 'SEC-US-0001'})
stl = call_settlement_tool({'instruction_id': 'SI-DLV-1001'})
ref = call_reference_data_tool({'security_id': 'SEC-US-0001'})
trd = call_trade_netting_tool({'account_id': 'ACC-DLV-001', 'security_id': 'SEC-US-0001'})

print('position records :', pos.record_count)
print('settlement records:', stl.record_count)
print('reference records :', ref.record_count)
print('trade_netting recs:', trd.record_count)
pos.records[0]

## 4. Run the full workflow

`run_workflow()` uses Google ADK if installed and otherwise falls back to a local deterministic runner. Either path produces the same `SessionState`.

In [ ]:
from settlement_agent.application.workflow import run_workflow, adk_available

print('ADK available?', adk_available())

state = run_workflow('SI-DLV-1001')
print('run_id:', state.run_id)
print('scenario:', state.classification.scenario_label)
print('reason_code:', state.diagnosis.reason_code)
print('is_firm_short:', state.diagnosis.is_firm_short)
print('approval:', state.approval.status, '| final?', state.is_final())

## 5. Inspect the evidence bundle

In [ ]:
ev = state.evidence
print('tools called:', ev.tools_called())
print('position rows :', len(ev.position))
print('settlement rows:', len(ev.settlement))
print('reference rows :', len(ev.reference))
print('trade rows     :', len(ev.trade_netting))

import json
print('\nposition[0] =', json.dumps(ev.position[0].model_dump(), indent=2))

## 6. Draft commentary

In [ ]:
print(state.commentary.text)
print('\nevidence_refs:', state.commentary.evidence_refs)
print('policy passed?', state.policy.passed)
print('policy findings:', state.policy.findings)

## 7. Capture human approval

Phase 1 supports `approved`, `rejected`, or `needs_edit`. The workflow does not finalise unless approval is captured. QMA send is **not** triggered automatically by this notebook.

In [ ]:
approval = 'approved'  # change to 'rejected' or 'needs_edit' to see other paths
reviewer = 'ops_analyst_demo'

final_state = run_workflow('SI-DLV-1001', approval_status=approval, reviewer=reviewer)
print('approval:', final_state.approval.status)
print('reviewer:', final_state.approval.reviewer)
print('decided_at:', final_state.approval.decided_at)
print('is_final:', final_state.is_final())

## 8. Run the eval suite

In [ ]:
from settlement_agent.application.eval_runner import run_eval_suite

results = run_eval_suite()
for r in results:
    status = 'PASS' if r.passed else 'FAIL'
    print(f'[{status}] {r.scenario_id}')
    if not r.passed:
        print('  checks:', r.checks)
        print('  notes :', r.notes)
passed = sum(1 for r in results if r.passed)
print(f'\n{passed}/{len(results)} eval cases passed')